# Kaggle Download Demo
This notebook shows how to call the data preparation package for a small Kaggle dataset.
The first implementation uses a tiny public dataset and verifies the file lands in the configured output folder.

In [ ]:
from pathlib import Path
import yaml

from data_prep.download import download_from_config, verify_download_path

# Always resolve the workspace root and keep downloads under the repo's staging area.
workspace_root = Path.cwd().resolve()
if workspace_root.name == "notebooks":
    workspace_root = workspace_root.parent

raw_dir = (workspace_root / "data" / "raw").resolve()
raw_dir.mkdir(parents=True, exist_ok=True)

config_dir = (workspace_root / "configs").resolve()
config_dir.mkdir(parents=True, exist_ok=True)

dataset_name = "healthcare-analytics-patient-flow-data"
dataset_id = "hassanjameelahmed/healthcare-analytics-patient-flow-data"
download_dir = (raw_dir / dataset_name).resolve()
download_dir.mkdir(parents=True, exist_ok=True)

# Update the config file in-place and default to CSV output.
config_path = (config_dir / "download_kaggle_sample.yaml").resolve()
with config_path.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f) or {}

config["destination"] = {
    "path": str(download_dir),
    "format": "csv",
}

with config_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False)

# Now download using the updated config.
result = download_from_config(config_path)
verify_download_path(result)
print(f"Download validated at: {result}")
print(f"Configured download directory: {download_dir}")
print(f"Expected format: csv")

Dataset URL: https://www.kaggle.com/datasets/hassanjameelahmed/healthcare-analytics-patient-flow-data


100%|██████████| 229k/229k [00:00<00:00, 266kB/s]


Download validated at: /home/rajiv/programming/kmds-dataset-util/data/raw/healthcare-analytics-patient-flow-data
Configured download directory: /home/rajiv/programming/kmds-dataset-util/data/raw/healthcare-analytics-patient-flow-data
Expected format: csv


In [ ]:
from pathlib import Path
import yaml

from data_prep.download import kaggle_url_to_config, download_from_config, verify_download_path

# Always resolve the workspace root and keep downloads under the repo's staging area.
workspace_root = Path.cwd().resolve()
if workspace_root.name == "notebooks":
    workspace_root = workspace_root.parent

raw_dir = (workspace_root / "data" / "raw").resolve()
raw_dir.mkdir(parents=True, exist_ok=True)

config_dir = (workspace_root / "configs").resolve()
config_dir.mkdir(parents=True, exist_ok=True)

dataset_url = "https://www.kaggle.com/datasets/sohier/seattle-police-department-911-incident-response"
dataset_id = "sohier/seattle-police-department-911-incident-response"
dataset_name = dataset_id.split("/")[-1]
download_dir = (raw_dir / dataset_name).resolve()
download_dir.mkdir(parents=True, exist_ok=True)

# Build the config file in the workspace config directory.
config_path = kaggle_url_to_config(
    dataset_url,
    write=True,
    output_dir=str(download_dir),
    config_dir=config_dir,
)

# Update the generated Kaggle config to use the workspace data directory and CSV output.
with config_path.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f) or {}

config["destination"] = {
    "path": str(download_dir),
    "format": "csv",
}

with config_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False)

# Use the configured download location.
result = download_from_config(config_path)
verify_download_path(result)
print(f"Download validated at: {result}")
print(f"Configured download directory: {download_dir}")
print(f"Expected format: csv")

Dataset URL: https://www.kaggle.com/datasets/sohier/seattle-police-department-911-incident-response


100%|██████████| 70.6M/70.6M [00:07<00:00, 9.41MB/s]



Download validated at: /home/rajiv/programming/kmds-dataset-util/data/raw/seattle-police-department-911-incident-response
Configured download directory: /home/rajiv/programming/kmds-dataset-util/data/raw/seattle-police-department-911-incident-response
Expected format: csv


In [17]:
from pathlib import Path
import inspect

from data_prep.llm import DatasetDictionaryBuilder

workspace_root = Path.cwd().resolve()
if workspace_root.name == "notebooks":
    workspace_root = workspace_root.parent

# Use the manually prepared raw dictionary for the healthcare dataset and keep the call
# compatible with older installed package versions in the notebook kernel.
dataset_dir = workspace_root / "data" / "raw" / "healthcare-analytics-patient-flow-data"
dictionary_path = dataset_dir / "data_dictionary_raw.csv"

if dataset_dir.exists() and dictionary_path.exists():
    builder = DatasetDictionaryBuilder(output_path=dictionary_path)
    method = builder.create_from_directory
    params = inspect.signature(method).parameters

    if "dictionary_path" in params:
        result = method(
            dataset_dir,
            output_path=dictionary_path,
            dictionary_path=dictionary_path,
            use_llm=False,
        )
    else:
        result = method(
            dataset_dir,
            output_path=dictionary_path,
            use_llm=False,
        )

    print(result)
    if "output_path" in result:
        print(f"Data dictionary generated at: {result['output_path']}")
else:
    print(f"Dataset or dictionary not found: {dataset_dir}")
    print("Download a dataset and confirm the raw data dictionary file is present, then rerun this cell.")

{'status': 'has_attribute_descriptions', 'attribute_descriptions': [{'column_name': 'Patient Id', 'description': 'Unique anonymized patient identifier', 'data_type': 'unknown'}, {'column_name': 'Patient Admission Date', 'description': 'Date of patient arrival/registration (MM/DD/YYYY)', 'data_type': 'unknown'}, {'column_name': 'Patient Admission Time', 'description': 'Time of patient arrival (HH:MM:SS AM/PM)', 'data_type': 'unknown'}, {'column_name': 'Patient_Admission_DateTime', 'description': 'Data quality issue: Appears to be corrupted name field; recommended to drop', 'data_type': 'unknown'}, {'column_name': 'Patient Gender', 'description': 'Biological sex: Male/Female', 'data_type': 'unknown'}, {'column_name': 'Patient Age', 'description': 'Patient age in years (range: 0-79)', 'data_type': 'unknown'}, {'column_name': 'Patient Race', 'description': 'Self-reported ethnicity/background', 'data_type': 'unknown'}, {'column_name': 'Department Referral', 'description': 'Specialty departm

In [18]:
import pandas as pd

hdd = pd.read_csv(dictionary_path)
hdd.head()

,column_name,description
0,Patient Id,Unique anonymized patient identifier
1,Patient Admission Date,Date of patient arrival/registration (MM/DD/YYYY)
2,Patient Admission Time,Time of patient arrival (HH:MM:SS AM/PM)
3,Patient_Admission_DateTime,Data quality issue: Appears to be corrupted na...
4,Patient Gender,Biological sex: Male/Female


In [19]:
from pathlib import Path
import pandas as pd

# Normalize the schema expected by downstream tooling.
if "column_name" in hdd.columns:
    hdd = hdd.rename(columns={"column_name": "attribute"})
elif "attribute" not in hdd.columns:
    raise ValueError("The dictionary must contain an 'attribute' or 'column_name' column.")

# Fix the column name typo and rename the merged field.
hdd["attribute"] = hdd["attribute"].replace({"Paitent Id": "Patient Id"})
hdd["attribute"] = hdd["attribute"].replace({"Merged": "Patient_Admission_DateTime"})

# Add Python type metadata for every column.
type_map = {
    "Patient Id": "str",
    "Patient Admission Date": "datetime.date",
    "Patient Admission Time": "pd.Timestamp",
    "Patient_Admission_DateTime": "pd.Timestamp",
    "Patient Gender": "str",
    "Patient Age": "int",
    "Patient Race": "str",
    "Department Referral": "str",
    "Patient Admission Flag": "bool",
    "Patient Satisfaction Score": "float",
}

hdd["python_type"] = hdd["attribute"].map(type_map).fillna("str")

# Keep the updated dictionary in the expected schema and write it back to disk.
hdd = hdd[["attribute", "description", "python_type"]]
hdd.to_csv(dictionary_path, index=False)

print(f"Updated dictionary written to: {dictionary_path}")
print(hdd)

Updated dictionary written to: /home/rajiv/programming/kmds-dataset-util/data/raw/healthcare-analytics-patient-flow-data/data_dictionary_raw.csv
                    attribute  \
0                  Patient Id   
1      Patient Admission Date   
2      Patient Admission Time   
3  Patient_Admission_DateTime   
4              Patient Gender   
5                 Patient Age   
6                Patient Race   
7         Department Referral   
8      Patient Admission Flag   
9  Patient Satisfaction Score   

                                         description    python_type  
0               Unique anonymized patient identifier            str  
1  Date of patient arrival/registration (MM/DD/YYYY)  datetime.date  
2           Time of patient arrival (HH:MM:SS AM/PM)   pd.Timestamp  
3  Data quality issue: Appears to be corrupted na...   pd.Timestamp  
4                        Biological sex: Male/Female            str  
5                 Patient age in years (range: 0-79)            int  
6

In [21]:
import pandas as pd
from pathlib import Path

# Use the downloaded Seattle CSV and the raw dictionary in the same folder.
download_root = Path(download_dir)

dictionary_path = download_root / "data_dictionary_raw.csv"

# Canonical data dictionary for the Seattle Police Department 911 incident response dataset.
rows = [
    {"attribute": "cad_event_number", "description": "Unique CAD event identifier assigned to the incident when it was logged into the dispatch system.", "python_type": "str"},
    {"attribute": "call_type", "description": "Method by which the CAD system received the event (for example, emergency call, telephone, or officer on-view).", "python_type": "str"},
    {"attribute": "priority", "description": "Operational priority level assigned to the call by the dispatch system.", "python_type": "str"},
    {"attribute": "initial_call_type", "description": "How the emergency incident was originally classified by the Communications Center when the call was first logged.", "python_type": "str"},
    {"attribute": "final_call_type", "description": "Official final case type logged when the responding unit cleared the incident.", "python_type": "str"},
    {"attribute": "cad_event_original_time_queued", "description": "Date and time the emergency call was initially received or generated in the CAD system.", "python_type": "pd.Timestamp"},
    {"attribute": "cad_event_arrived_time", "description": "Date and time the first responding police unit arrived at the scene.", "python_type": "pd.Timestamp"},
    {"attribute": "dispatch_precinct", "description": "The overarching police precinct responsible for the response (for example, North, South, West, East, or Southwest).", "python_type": "str"},
    {"attribute": "dispatch_sector", "description": "The operational sub-district or sector within the precinct where the unit was assigned.", "python_type": "str"},
    {"attribute": "dispatch_beat", "description": "The specific geographic patrol beat to which the call was assigned.", "python_type": "str"},
    {"attribute": "dispatch_longitude", "description": "Longitude coordinate for the incident location. In some public versions, values may be blurred or truncated to the 100-block level to protect privacy.", "python_type": "float"},
    {"attribute": "dispatch_latitude", "description": "Latitude coordinate for the incident location. In some public versions, values may be blurred or truncated to the 100-block level to protect privacy.", "python_type": "float"},
    {"attribute": "dispatch_reporting_area", "description": "The micro-level geographic reporting area used by the department for operational tracking.", "python_type": "str"},
    {"attribute": "cad_event_clearance_description", "description": "Official disposition or clearance code/description recorded when the incident was resolved or closed (for example, Report Written, Citation, Assistance Rendered, or Canceled).", "python_type": "str"},
    {"attribute": "cad_event_response_category", "description": "Category indicating who managed the unit dispatch, such as Seattle Police Department (SPD), CARE, or a joint SPD/CARE co-response.", "python_type": "str"},
]

dictionary_df = pd.DataFrame(rows, columns=["attribute", "description", "python_type"])
dictionary_df.to_csv(dictionary_path, index=False)

print(f"Updated Seattle 911 data dictionary written to: {dictionary_path}")
print(dictionary_df)


Updated Seattle 911 data dictionary written to: /home/rajiv/programming/kmds-dataset-util/data/raw/seattle-police-department-911-incident-response/data_dictionary_raw.csv
                          attribute  \
0                  cad_event_number   
1                         call_type   
2                          priority   
3                 initial_call_type   
4                   final_call_type   
5    cad_event_original_time_queued   
6            cad_event_arrived_time   
7                 dispatch_precinct   
8                   dispatch_sector   
9                     dispatch_beat   
10               dispatch_longitude   
11                dispatch_latitude   
12          dispatch_reporting_area   
13  cad_event_clearance_description   
14      cad_event_response_category   

                                          description   python_type  
0   Unique CAD event identifier assigned to the in...           str  
1   Method by which the CAD system received the ev...           